# KITScenes-LongTail World Modeling Challenge — End-to-End Kaggle Pipeline
### NeurIPS 2026 Workshop on World Models in Physical AI

This production-ready notebook generates multi-view surround driving videos for all **400 test scenarios** (`0000000` to `0000399`) using **Stable Video Diffusion (SVD)** conditioned on observed $t=0$ camera frames and planned trajectories, then packages them for submission to the official [Hugging Face Challenge Space](https://huggingface.co/spaces/KIT-MRT/KITScenes-LongTail-Challenge).

### Key Challenge Requirements & Scoring:
- **6 Surround Cameras**: `front` (required), `front_left`, `front_right`, `rear`, `rear_left`, `rear_right`.
- **Video Encoding Contract**:
  - Format: **H.264** in `.mp4` container.
  - Dimensions: **876x568** resolution (exact KITScenes aspect ratio).
  - Frames: **Exactly 25 frames** @ **5.0 fps** (spanning 0.2 s to 5.0 s into the future).
  - Size: **<= 50 MiB uncompressed per video**, total archive `videos.zip` **<= 3.0 GiB**.
- **Leaderboard Ranking Metrics**:
  - **Perceptual Quality (PQ)**: Primary ranking metric evaluated via Q-Align (`front` weighted 50%, side cameras 10% each).
  - **Ego-Motion Coherence (EMC)**: Tie-breaker metric computed via optical flow across surround cameras.
  - **Multi-Maneuver Score (MMS)**: Evaluated against reference trajectories on the private evaluation server.
- **Kaggle Hardware Optimization**: Tuned for 2x Nvidia T4 16GB GPUs with FP16 precision, memory-efficient attention, CPU offloading, and fast multi-threaded H.264 encoding.


## Step 1: Environment Setup & Hardware Verification
Detect GPUs, configure PyTorch, verify `ffmpeg` / `ffprobe`, and install necessary dependencies.

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

# Sync or clone kitscenes package from GitHub
print("Syncing kitscenes package from GitHub...")
subprocess.run(["rm", "-rf", "/tmp/repo_src"], check=False)
subprocess.run(["git", "clone", "https://github.com/wageshsharma20/kitscenes-world-model.git", "/tmp/repo_src"], check=True)
for item in ["kitscenes", "scripts", "validate_submission.py"]:
    subprocess.run(["rm", "-rf", item], check=False)
    subprocess.run(["cp", "-r", f"/tmp/repo_src/{item}", "."], check=True)
subprocess.run(["rm", "-rf", "/tmp/repo_src"], check=False)
print("kitscenes package ready and up to date.")

# Add workspace root to sys.path
for candidate in [Path("."), Path("/kaggle/working"), Path("/kaggle/working/kitscenes-world-model")]:
    if candidate.exists() and str(candidate.resolve()) not in sys.path:
        sys.path.insert(0, str(candidate.resolve()))

print(f"Python Version: {sys.version}")

# Check GPU Availability
try:
    import torch
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        num_gpus = torch.cuda.device_count()
        print(f"Detected {num_gpus} GPU(s):")
        for i in range(num_gpus):
            props = torch.cuda.get_device_properties(i)
            print(f"  [GPU {i}] {props.name} ({props.total_memory / 1024**3:.2f} GB VRAM)")
except ImportError:
    print("PyTorch not installed yet.")

# Verify or install ffmpeg
ffmpeg_bin = shutil.which("ffmpeg")
if not ffmpeg_bin:
    print("Installing ffmpeg...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)

print(f"ffmpeg:  {shutil.which('ffmpeg')}")
print(f"ffprobe: {shutil.which('ffprobe')}")


In [ ]:
# Install required libraries for world modeling & video processing
!pip install -q diffusers accelerate huggingface_hub tqdm Pillow pyarrow


## Step 2: Hugging Face Authentication & Directory Setup
Authenticate using Kaggle Secrets (`HF_TOKEN`) and set up the directory structure.

In [ ]:
from huggingface_hub import login

# Retrieve token from Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN)
hf_token = ""
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("Retrieved HF_TOKEN from Kaggle Secrets.")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")

if hf_token:
    login(token=hf_token)
    print("Authenticated successfully with Hugging Face Hub.")
else:
    print("Warning: Running without Hugging Face token (gated dataset access requires token).")

# Setup working directories
BASE_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".").resolve()
DATA_DIR = BASE_DIR / "data"
OBSERVED_DIR = DATA_DIR / "observed_t0"
RENDERS_DIR = BASE_DIR / "renders"
SUBMISSIONS_DIR = BASE_DIR / "submissions"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OBSERVED_DIR.mkdir(parents=True, exist_ok=True)
RENDERS_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory:        {DATA_DIR}")
print(f"Observed t=0 frames:   {OBSERVED_DIR}")
print(f"Renders directory:     {RENDERS_DIR}")
print(f"Submissions directory: {SUBMISSIONS_DIR}")


## Step 3: Trajectory Verification & MMS Sentinel Guard
Load and validate the trajectory plan (`submissions/trajectories.jsonl`).

> [!NOTE]
> In the official test split, ground-truth future reference paths (`expert_like`, etc.) are redacted to sentinel values `[[-100.0, -100.0]]` to prevent test leakage. True MMS is scored exclusively on the Hugging Face server.

In [ ]:
from kitscenes import spec, trajectories

trajectories_file = SUBMISSIONS_DIR / "trajectories.jsonl"

# If trajectories.jsonl is not already present, build using the tactical planner
if not trajectories_file.exists():
    hist_file = DATA_DIR / "history.jsonl"
    inst_file = DATA_DIR / "instructions.jsonl"
    if not hist_file.exists():
        print("Streaming split metadata (odometry & instructions) from Hugging Face...")
        subprocess.run([
            sys.executable, "scripts/download_dataset.py",
            "--metadata-only",
            "--output-dir", str(DATA_DIR),
            "--token", hf_token,
        ], check=True)
    print(f"Planning trajectories from {hist_file}...")
    from kitscenes import vlm_planner
    vlm_planner.plan_vlm_file(
        source=hist_file,
        destination=trajectories_file,
        instructions_path=inst_file,
        heuristic_fallback=True,
    )

# Validate trajectories against schema
traj_report = trajectories.inspect_trajectories(trajectories_file)
print(f"Trajectories validated: {traj_report.count}/{spec.SCENARIO_COUNT} records")
print(f"Errors: {len(traj_report.errors)}, Warnings: {len(traj_report.warnings)}")
assert traj_report.ok, f"Trajectory validation failed: {traj_report.errors}"


## Step 4: Observed $t=0$ Conditioning Frame Extraction
Generative world models require the real observed $t=0$ image for all 6 cameras to maintain visual coherence with the past driving scene.
On Kaggle (73 GB disk), we download the test split parquet file and extract the 6 downscaled ($876\times 568$) JPEG frames for all 400 scenarios (only ~190 MB total).

In [ ]:
from huggingface_hub import hf_hub_download

parquet_path = DATA_DIR / "test-00000-of-00001.parquet"

# Check how many scenarios have already been extracted
existing_t0 = [
    d.name for d in OBSERVED_DIR.iterdir()
    if d.is_dir() and len(list(d.glob("*.jpg"))) == 6
]
print(f"Existing t=0 conditioning scenarios: {len(existing_t0)}/{spec.SCENARIO_COUNT}")

if len(existing_t0) < spec.SCENARIO_COUNT:
    if not parquet_path.exists():
        print("Downloading test parquet (~32 GB) to local Kaggle disk...")
        hf_hub_download(
            repo_id="KIT-MRT/KITScenes-LongTail",
            repo_type="dataset",
            filename="data/test-00000-of-00001.parquet",
            token=hf_token,
            local_dir=DATA_DIR,
        )
        print("Download complete.")

    print("Extracting t=0 frames for all 400 scenarios (downscaled to 876x568)...")
    subprocess.run([
        sys.executable, "scripts/extract_t0_frames.py",
        "--parquet", str(parquet_path),
        "--output-dir", str(OBSERVED_DIR),
        "--resize",
    ], check=True)
    print("Observed t=0 extraction finished.")


## Step 5: GPU Single-Scenario Verification (Scenario 0000902, Front Camera)
Execute genuine pretrained SVD inference on 1 T4 GPU for Scenario 0000902 (front camera, 25 frames).
Records real hardware metrics: peak allocated/reserved VRAM, initialization latency, inference duration per frame, and frame divergence vs t=0.

In [ ]:
# Run genuine pretrained SVD single-camera inference on 1 T4 GPU
!python scripts/run_kaggle_svd_single_camera.py --device cuda --steps 15


## Step 6: Multi-View World Model Generation Loop
Generate all 2,400 videos (400 scenarios $\times$ 6 surround cameras):
- **SVD Diffusion World Model (`BACKBONE = 'svd'`)**: Conditioned on the 6 observed $t=0$ camera frames and trajectory forward speed.
- **Geometric Baseline (`BACKBONE = 'geometric'`)**: Fast procedural renderer for quick dry-runs.

In [ ]:
import json
import time
from tqdm.auto import tqdm
from kitscenes import video_engine

# Pipeline Configuration
BACKBONE = "svd"  # 'svd' for high perceptual quality on GPU; 'geometric' for quick test
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load planned trajectories
trajectories_dict = {}
with open(trajectories_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rec = json.loads(line)
            trajectories_dict[rec["scenario_id"]] = rec["future_trajectory"]

print(f"Loaded {len(trajectories_dict)} trajectories.")
print(f"Initializing WorldModelPipeline with backbone='{BACKBONE}', device='{DEVICE}'...")

encoder = video_engine.VideoEncoder(crf=24, preset="fast")
rig = video_engine.CameraRig()

if BACKBONE == "svd":
    generator = video_engine.StableVideoDiffusionGenerator(device=DEVICE)
else:
    generator = video_engine.GeometricWorldModel(rig)

def render_and_encode_scenario(sc_id: str) -> str:
    """Generate 6 camera videos for a scenario and encode directly to disk."""
    traj = trajectories_dict[sc_id]
    sc_dir = RENDERS_DIR / sc_id
    sc_dir.mkdir(parents=True, exist_ok=True)

    cond_images = None
    if BACKBONE == "svd":
        cond_images = video_engine.load_conditioning_images(OBSERVED_DIR, sc_id)

    videos = generator.generate_scenario(
        scenario_id=sc_id,
        trajectory=traj,
        conditioning_images=cond_images,
    )

    for cam_name, frames in videos.items():
        out_path = sc_dir / f"{cam_name}.mp4"
        encoder.encode(frames, out_path, validate=False)
    return sc_id

# Run batch generation across all 400 test scenarios
total_scenarios = len(spec.SCENARIO_IDS)
print(f"Generating 6 cameras x {total_scenarios} scenarios = {total_scenarios * 6} videos...")
start_time = time.time()

for sc_id in tqdm(spec.SCENARIO_IDS, desc=f"Rendering ({BACKBONE})"):
    render_and_encode_scenario(sc_id)

duration = time.time() - start_time
print(f"Finished rendering {total_scenarios} scenarios in {duration:.1f}s ({duration/total_scenarios:.2f}s / scenario).")


## Step 7: Packaging & Server-Equivalent Validation
Package the rendered videos into `videos.zip` and run `validate_submission.py` to confirm 0 errors against official competition rules.

In [ ]:
from kitscenes import archive

zip_output = SUBMISSIONS_DIR / "videos.zip"
print(f"Packaging rendered videos into {zip_output}...")

pack_report = archive.build_archive(
    directory=RENDERS_DIR,
    output=zip_output,
    allow_partial=False,
    check_media=True,
    jobs=8,
)

print("\n" + "=" * 60)
print("SUBMISSION ARCHIVE REPORT:")
for line in pack_report.summary_lines():
    print("  *", line)
print(f"Errors:   {len(pack_report.errors)}")
print(f"Warnings: {len(pack_report.warnings)}")
print("=" * 60)

if pack_report.errors:
    print("Errors encountered:")
    for err in pack_report.errors[:10]:
        print("  !", err)
assert pack_report.ok, "Archive packaging failed validation!"


In [ ]:
# Run the comprehensive offline validator on both submission files
!python validate_submission.py \
    --videos {zip_output} \
    --trajectories {trajectories_file}


## Step 8: Ready for Leaderboard Submission!

The submission files are located at:
1. **`submissions/videos.zip`** (all 400 scenarios x 6 cameras, H.264, 876x568 @ 5 fps, < 3 GiB).
2. **`submissions/trajectories.jsonl`** (400 scenario trajectories, 25 waypoints at 5 Hz).

To submit:
1. Download `videos.zip` and `trajectories.jsonl` from Kaggle's **Data / Output** tab.
2. Go to the [KITScenes-LongTail Challenge Space](https://huggingface.co/spaces/KIT-MRT/KITScenes-LongTail-Challenge).
3. Under the **Submit** tab, upload `videos.zip` and `trajectories.jsonl`.
4. Evaluation takes ~20–30 minutes.